In [1]:
import numpy as np

In [2]:
class NearestNeighbor(object):
  def __init__(self,k=3):
    self.k=k

  def train(self, X, y):
    """ X is N x D where each row is an example. Y is 1-dimension of size N """
    # the nearest neighbor classifier simply remembers all the training data
    self.Xtr = X
    self.ytr = y

  def predict(self, X):
    """ X is N x D where each row is an example we wish to predict label for """
    num_test = X.shape[0]
    # lets make sure that the output type matches the input type
    Ypred = np.zeros(num_test, dtype = self.ytr.dtype)

    # loop over all test rows
    for i in range(num_test):
      # find the nearest training image to the i'th test image
      # using the L2 distance (ignore sqrt to save a little time)
      distances = np.sum(np.square((self.Xtr - X[i,:])), axis = 1)
      k_indices = np.argpartition(distances,self.k)[:self.k] # get the index with k-smallest distance
      k_labels=self.ytr[k_indices]
      Ypred[i] = np.bincount(k_labels).argmax() # predict the label of the nearest example
    return Ypred



In [3]:
# 数据集根目录（确保末尾带斜杠）
p = "C:/Jupyter(Anaconda)/data/mnist/MNIST/raw/"

# 1. 读取图像二进制数据
# np.fromfile 读取整个文件为 uint8 一维数组，[16:] 切片跳过标准图像文件头（4字节魔数+4字节数量+4行+4列）
X_train_raw = np.fromfile(p + "train-images-idx3-ubyte", dtype=np.uint8)[16:]
X_test_raw  = np.fromfile(p + "t10k-images-idx3-ubyte",  dtype=np.uint8)[16:]

# 2. 读取标签二进制数据
# 标签文件头较短（4字节魔数+4字节数量），共8字节，用 [8:] 跳过
y_train_raw = np.fromfile(p + "train-labels-idx1-ubyte", dtype=np.uint8)[8:]
y_test_raw  = np.fromfile(p + "t10k-labels-idx1-ubyte",  dtype=np.uint8)[8:]

# 3. 维度重塑：将一维像素流转为 (样本数, 特征维度) 的二维矩阵
# MNIST 单张图 28x28=784 像素，-1 表示让 numpy 自动推算样本数量（训练集60000，测试集10000）
X_train = X_train_raw.reshape(-1, 784)
X_test  = X_test_raw.reshape(-1, 784)

# 4. 类型转换 & 🚨 归一化（KNN 距离计算必须步骤！）
# 原始像素为 0~255 的整数，直接计算距离会受绝对值尺度主导。除以 255.0 映射到 [0, 1] 浮点数区间
X_train = X_train.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0
y_train = y_train_raw.astype(np.int32)
y_test  = y_test_raw.astype(np.int32)
# 直接切片（仅保留前 N 个，速度最快。⚠️需确保原数据已打乱顺序）
N = 3000  # 修改为你想要的训练样本数量
# np.random.seed(42)  # 可选：固定随机种子，保证每次运行结果可复现
idx = np.random.permutation(len(X_train))  # 生成 0~N-1 的随机排列索引
X_train = X_train[idx[:N]]                 # 按打乱后的索引取前 N 行特征
y_train = y_train[idx[:N]]                 # 按相同索引取前 N 个标签

# 5. 验证输出格式是否匹配你的 NearestNeighbor 要求
print(f"训练集 -> X: {X_train.shape}, y: {y_train.shape}")
print(f"测试集 -> X: {X_test.shape},  y: {y_test.shape}")

训练集 -> X: (3000, 784), y: (3000,)
测试集 -> X: (10000, 784),  y: (10000,)


In [4]:
nn_5=NearestNeighbor(k=5)
nn_5.train(X_train, y_train)
nn_7=NearestNeighbor(k=7)
nn_7.train(X_train, y_train)


In [5]:
y_pred=nn_5.predict(X_test)
acc = np.mean(y_pred == y_test)
print ('accuracy: %f' % (acc,))

accuracy: 0.922800


In [ ]:
y_pred_7=nn_7.predict(X_test)
acc_ = np.mean(y_pred_7 == y_test)
print ('accuracy: %f' % (acc_,))